# 01C. Signal Discovery Search Space

Purpose: define a machine-readable search-space blueprint for future automatic signal candidate generation in Notebook 02. This notebook is planning/config only: it does not generate candidate signals, score signals, or modify downstream validation logic.

## 1. Scope Boundaries

- Writes a structured discovery search space to SQLite.
- Defines formula families, template names, parameter grids, transforms, direction hypotheses, and research notes.
- Does not modify Notebook 02, candidate signal formulas, scoring, validation, WFV, decay, regime, health, reproducibility, diversity, alpha, stress, survivor, portfolio, or ML logic.

## 2. Imports and Config

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import get_db_path, load_table, table_exists
from src.run_config import make_run_id, make_run_timestamp
from src.signal_discovery_space import (
    SIGNAL_DISCOVERY_SPACE_TABLES,
    build_signal_discovery_search_space,
    save_signal_discovery_search_space,
)

DB_PATH = get_db_path()
SIGNAL_DISCOVERY_SPACE_VERSION = "phase2_signal_discovery_space_v1"

pd.set_option("display.max_columns", 200)
DB_PATH

PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

## 3. Create run_id / timestamp

In [2]:
run_id = make_run_id(prefix="phase2_signal_discovery_space")
run_timestamp = make_run_timestamp()

run_id, run_timestamp

('phase2_signal_discovery_space_20260503_204034', '2026-05-03 20:40:34')

## 4. Build Search Space

In [3]:
signal_discovery_search_space = build_signal_discovery_search_space()

family_counts = (
    signal_discovery_search_space["discovery_family"]
    .value_counts(dropna=False)
    .rename_axis("discovery_family")
    .reset_index(name="n_templates")
    .sort_values("discovery_family")
)
priority_counts = (
    signal_discovery_search_space["priority"]
    .value_counts(dropna=False)
    .rename_axis("priority")
    .reset_index(name="n_templates")
)

print(f"Discovery search-space rows: {len(signal_discovery_search_space)}")
print(f"Discovery families: {signal_discovery_search_space['discovery_family'].nunique()}")
display(family_counts)
display(priority_counts)
display(signal_discovery_search_space)

Discovery search-space rows: 8
Discovery families: 8


,discovery_family,n_templates
1,beta_neutral_return,1
6,correlation_change,1
0,cross_sectional_relative_return,1
7,liquidity_adjusted_return,1
5,reversal_overextension,1
2,volatility_adjusted_momentum,1
3,volatility_surprise,1
4,volume_return_interaction,1


,priority,n_templates
0,HIGH,4
1,MEDIUM,4


,discovery_family,base_formula,signal_template_name,parameter_grid,required_inputs,transform_options,direction_hypotheses,expected_horizons,expected_diversification_role,priority,notes
0,cross_sectional_relative_return,close.pct_change(window) - cross_sectional_mea...,relative_return_{window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Adds same-date universe-relative price informa...,HIGH,Use trailing returns only; center by same-date...
1,beta_neutral_return,close.pct_change(window) - rolling_beta(beta_w...,beta_neutral_return_{window}_{beta_window},"{""beta_windows"":[60,120],""directions"":[""positi...","[""close"",""benchmark_close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Separates stock-specific return from broad sys...,HIGH,Use SPY when present; otherwise equal-weight u...
2,volatility_adjusted_momentum,close.pct_change(window) / rolling_std(daily_r...,vol_adj_momentum_{window}_{vol_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""10d"",""20d""]",Tests smoother momentum variants without relyi...,MEDIUM,Handle zero realized volatility as missing bef...
3,volatility_surprise,"rolling_std(daily_return, short_window) / roll...",vol_surprise_{short_window}_{long_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Captures volatility regime transitions rather ...,MEDIUM,Treat as volatility-adjacent and require diver...
4,volume_return_interaction,"return(window) combined with volume trend, acc...",volume_return_interaction_{window}_{interaction},"{""directions"":[""positive_edge"",""negative_edge_...","[""close"",""volume""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Adds participation context to price moves and ...,HIGH,Use trailing volume windows only; replace zero...
5,reversal_overextension,"negative return(window), distance from moving ...",reversal_overextension_{window}_{measure},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""1d"",""5d"",""10d""]",Expands short-horizon pullback behavior beyond...,MEDIUM,Keep formulas trailing-only; execution lag rem...
6,correlation_change,"rolling_corr(stock_return, benchmark_return, s...",corr_change_{short_window}_{long_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close"",""benchmark_close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""10d"",""20d""]",Targets changing market co-movement and crowdi...,HIGH,"Use SPY if available, otherwise equal-weight m..."
7,liquidity_adjusted_return,return(window) scaled or conditioned by dollar...,liquidity_adjusted_return_{window}_{liquidity_...,"{""directions"":[""positive_edge"",""negative_edge_...","[""close"",""volume""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Separates return effects that depend on tradab...,MEDIUM,Guard dollar-volume denominators and keep liqu...


## 5. Validate JSON Fields

In [4]:
json_columns = [
    "parameter_grid",
    "required_inputs",
    "transform_options",
    "direction_hypotheses",
    "expected_horizons",
]

json_validation = []
for column in json_columns:
    parsed = signal_discovery_search_space[column].map(json.loads)
    json_validation.append(
        {
            "column": column,
            "n_rows": len(parsed),
            "n_valid_json": int(parsed.notna().sum()),
            "example": parsed.iloc[0],
        }
    )

json_validation = pd.DataFrame(json_validation)
display(json_validation)

,column,n_rows,n_valid_json,example
0,parameter_grid,8,8,"{'directions': ['positive_edge', 'negative_edg..."
1,required_inputs,8,8,[close]
2,transform_options,8,8,"[raw, rank, zscore, winsorized_zscore]"
3,direction_hypotheses,8,8,"[positive_edge, negative_edge_reverse]"
4,expected_horizons,8,8,"[5d, 10d, 20d]"


## 6. Save Search Space to SQLite

In [5]:
saved_paths = save_signal_discovery_search_space(
    search_space=signal_discovery_search_space,
    db_path=DB_PATH,
    run_id=run_id,
    search_space_version=SIGNAL_DISCOVERY_SPACE_VERSION,
    timestamp=run_timestamp,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "current_table": tables[0],
            "history_table": tables[1],
            "sqlite_path": str(saved_paths[artifact]),
        }
        for artifact, tables in SIGNAL_DISCOVERY_SPACE_TABLES.items()
    ]
)

display(sqlite_tables_written)

,artifact,current_table,history_table,sqlite_path
0,search_space,signal_discovery_search_space_current,signal_discovery_search_space_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 7. Read Back and Summarize

In [6]:
current_table = SIGNAL_DISCOVERY_SPACE_TABLES["search_space"][0]
if not table_exists(current_table, db_path=DB_PATH):
    raise ValueError(f"Expected SQLite table missing: {current_table}")

search_space_current = load_table(current_table, db_path=DB_PATH)

final_summary = pd.DataFrame(
    [
        {"metric": "run_id", "value": run_id},
        {"metric": "run_timestamp", "value": run_timestamp},
        {"metric": "signal_discovery_space_version", "value": SIGNAL_DISCOVERY_SPACE_VERSION},
        {"metric": "search_space_rows", "value": len(search_space_current)},
        {"metric": "discovery_family_count", "value": search_space_current["discovery_family"].nunique()},
        {"metric": "sqlite_current_table", "value": current_table},
    ]
)

print("Search-space current table")
display(search_space_current)

print("Family counts")
display(family_counts)

print("SQLite tables written")
display(sqlite_tables_written)

print("Final summary")
display(final_summary)

Search-space current table


,discovery_family,base_formula,signal_template_name,parameter_grid,required_inputs,transform_options,direction_hypotheses,expected_horizons,expected_diversification_role,priority,notes,run_id,search_space_version,timestamp
0,cross_sectional_relative_return,close.pct_change(window) - cross_sectional_mea...,relative_return_{window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Adds same-date universe-relative price informa...,HIGH,Use trailing returns only; center by same-date...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
1,beta_neutral_return,close.pct_change(window) - rolling_beta(beta_w...,beta_neutral_return_{window}_{beta_window},"{""beta_windows"":[60,120],""directions"":[""positi...","[""close"",""benchmark_close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Separates stock-specific return from broad sys...,HIGH,Use SPY when present; otherwise equal-weight u...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
2,volatility_adjusted_momentum,close.pct_change(window) / rolling_std(daily_r...,vol_adj_momentum_{window}_{vol_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""10d"",""20d""]",Tests smoother momentum variants without relyi...,MEDIUM,Handle zero realized volatility as missing bef...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
3,volatility_surprise,"rolling_std(daily_return, short_window) / roll...",vol_surprise_{short_window}_{long_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Captures volatility regime transitions rather ...,MEDIUM,Treat as volatility-adjacent and require diver...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
4,volume_return_interaction,"return(window) combined with volume trend, acc...",volume_return_interaction_{window}_{interaction},"{""directions"":[""positive_edge"",""negative_edge_...","[""close"",""volume""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""5d"",""10d"",""20d""]",Adds participation context to price moves and ...,HIGH,Use trailing volume windows only; replace zero...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
5,reversal_overextension,"negative return(window), distance from moving ...",reversal_overextension_{window}_{measure},"{""directions"":[""positive_edge"",""negative_edge_...","[""close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""1d"",""5d"",""10d""]",Expands short-horizon pullback behavior beyond...,MEDIUM,Keep formulas trailing-only; execution lag rem...,phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
6,correlation_change,"rolling_corr(stock_return, benchmark_return, s...",corr_change_{short_window}_{long_window},"{""directions"":[""positive_edge"",""negative_edge_...","[""close"",""benchmark_close""]","[""raw"",""rank"",""zscore"",""winsorized_zscore""]","[""positive_edge"",""negative_edge_reverse""]","[""10d"",""20d""]",Targets changing market co-movement and crowdi...,HIGH,"Use SPY if available, otherwise equal-weight m...",phase2_signal_discovery_space_20260503_204034,phase2_signal_discovery_space_v1,2026-05-03 20:40:34
7,liquidity_adjusted_return,return(window) scaled or conditioned by dollar...,liqui

Family counts


,discovery_family,n_templates
1,beta_neutral_return,1
6,correlation_change,1
0,cross_sectional_relative_return,1
7,liquidity_adjusted_return,1
5,reversal_overextension,1
2,volatility_adjusted_momentum,1
3,volatility_surprise,1
4,volume_return_interaction,1


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,search_space,signal_discovery_search_space_current,signal_discovery_search_space_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


Final summary


,metric,value
0,run_id,phase2_signal_discovery_space_20260503_204034
1,run_timestamp,2026-05-03 20:40:34
2,signal_discovery_space_version,phase2_signal_discovery_space_v1
3,search_space_rows,8
4,discovery_family_count,8
5,sqlite_current_table,signal_discovery_search_space_current
